In [ ]:
# === Mount Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

# === Install necessary packages (only run once) ===
!pip install -q mne plotly colorama pyphen

# === Imports ===
import os
import gc
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import pyphen
!pip install mne
import mne
import plotly.graph_objects as go
from IPython.display import display
from colorama import Fore, Style, init

# === Colorama init (for colored terminal printouts) ===
init(autoreset=True)

# LAG 20

In [ ]:
import os
import numpy as np
import librosa
import mne
from scipy.signal import hilbert, resample
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

# === Paths ===
stim_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli"
eeg_base_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData"

# === Parameters ===
sfreq = 250  # Sampling rate (Hz)

# --- Lags: -0.5 to 1.5 seconds ---
lags = np.arange(int(-0.5 * sfreq), int(1.5 * sfreq) + 1)

# === Adaptive Alpha Search (Initial broad range) ===
initial_alphas = np.logspace(-2, 4, 10)  # 10 values from 10^-2 to 10^4

# === Helper: Lagged Matrix ===
def make_lagged_matrix(x, lags):
    max_lag = np.max(np.abs(lags))
    padded = np.pad(x, (max_lag,), mode='constant')
    return np.stack([
        padded[max_lag + lag: max_lag + lag + len(x)]
        for lag in lags
    ], axis=1)

# === Loop through participants ===
for participant in range(11, 28):  # 11 to 27
    eeg_path = f"{eeg_base_dir}/{participant}/{participant}_O1_epo_clean.fif"

    if not os.path.exists(eeg_path):
        print(f"⚠️ Skipping Participant {participant} (file not found).")
        continue

    print(f"\n===== Processing Participant {participant} =====")

    # === Load EEG ===
    epochs = mne.read_epochs(eeg_path, preload=True)
    epochs.resample(sfreq)
    eeg_data = epochs.get_data()
    eeg_data = np.mean(eeg_data, axis=1)  # Average channels
    n_epochs, n_times = eeg_data.shape

    # === Load and process audio envelopes ===
    wav_files = sorted([f for f in os.listdir(stim_dir) if f.endswith(".wav")])
    if len(wav_files) < n_epochs:
        print(f"⚠️ Only {len(wav_files)} audio files found. Truncating EEG data.")
        eeg_data = eeg_data[:len(wav_files)]
        n_epochs = len(eeg_data)

    envelopes = []
    for f in wav_files[:n_epochs]:
        y, sr = librosa.load(os.path.join(stim_dir, f), sr=None)
        y_resampled = resample(y, int(len(y) * sfreq / sr))
        env = np.abs(hilbert(y_resampled))
        envelopes.append(env[:n_times])

    n_trials = min(len(envelopes), len(eeg_data))
    envelopes = envelopes[:n_trials]
    eeg_data = eeg_data[:n_trials]

    best_alphas = []
    trial_r2_scores = []

    # === Step 1: Find best alpha per trial ===
    for trial in range(n_trials):
        X = make_lagged_matrix(envelopes[trial], lags)
        Y = eeg_data[trial]

        min_len = min(X.shape[0], len(Y))
        X = X[:min_len]
        Y = Y[:min_len]

        if trial == 0:
            print(f"🔍 Diagnostic: {X.shape[0]} samples × {X.shape[1]} predictors (lags)")

        X_std = StandardScaler().fit_transform(X)
        Y_std = StandardScaler().fit_transform(Y.reshape(-1, 1)).ravel()

        best_r2 = -np.inf
        best_alpha = None

        # Broad alpha search
        for alpha in initial_alphas:
            model = Ridge(alpha=alpha)
            model.fit(X_std, Y_std)
            r2 = r2_score(Y_std, model.predict(X_std))
            if r2 > best_r2:
                best_r2 = r2
                best_alpha = alpha

        # Refined search around best alpha
        refined_alphas = np.logspace(
            np.log10(best_alpha) - 0.5,
            np.log10(best_alpha) + 0.5,
            10
        )
        for alpha in refined_alphas:
            model = Ridge(alpha=alpha)
            model.fit(X_std, Y_std)
            r2 = r2_score(Y_std, model.predict(X_std))
            if r2 > best_r2:
                best_r2 = r2
                best_alpha = alpha

        best_alphas.append(best_alpha)
        trial_r2_scores.append(best_r2)

    # === Step 2: Select participant-level alpha ===
    participant_alpha = np.median(best_alphas)
    print(f"✅ Participant {participant}: Median best alpha = {participant_alpha:.2f}")
    print(f"✅ Participant {participant}: Mean trial R² = {np.mean(trial_r2_scores):.4f}")

    # === Step 3: Refit TRF using participant alpha ===
    participant_trfs = []
    for trial in range(n_trials):
        X = make_lagged_matrix(envelopes[trial], lags)
        Y = eeg_data[trial]

        min_len = min(X.shape[0], len(Y))
        X = X[:min_len]
        Y = Y[:min_len]

        X_std = StandardScaler().fit_transform(X)
        Y_std = StandardScaler().fit_transform(Y.reshape(-1, 1)).ravel()

        model = Ridge(alpha=participant_alpha)
        model.fit(X_std, Y_std)
        participant_trfs.append(model.coef_)

    # === Plot TRF (single TRF per participant) ===
    time_lags_ms = lags * (1000 / sfreq)
    mean_trf = np.mean(participant_trfs, axis=0)
    plt.figure(figsize=(10, 4))
    plt.plot(time_lags_ms, mean_trf, label="TRF")
    plt.axvline(0, color='k', linestyle='--', label="Stimulus onset")
    plt.title(f"Participant {participant} - mTRF (-0.5 to 1.5 s)")
    plt.xlabel("Lag (ms)")
    plt.ylabel("TRF Weight")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # === Plot Best Alpha Values per Trial ===
    plt.figure(figsize=(8, 4))
    plt.plot(best_alphas, marker='o')
    plt.xlim(0, len(best_alphas))
    plt.yscale("log")
    plt.xlabel("Trial")
    plt.ylabel("Best Alpha (log scale)")
    plt.title(f"Participant {participant} - Best Alpha per Trial")
    plt.grid(True, which="both")
    plt.show()
